# CFB intro — sportsdataverse-py

ESPN-backed college football: play-by-play, schedule, teams, and per-play participant resolution. Wrappers follow the `espn_cfb_*` pattern; pre-built datasets load via `load_cfb_*`.

R companion: [cfbfastR](https://cfbfastR.sportsdataverse.org) — the same verbs in R. Part of the [SportsDataverse](https://py.sportsdataverse.org/docs/ecosystem).

## Setup

```sh
pip install sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse as sdv

## Teams

In [ ]:
teams = sdv.cfb.espn_cfb_teams()
teams.shape

In [ ]:
teams.select(['team_id', 'team_location', 'team_name', 'team_abbreviation']).head()

## Schedule

In [ ]:
schedule = sdv.cfb.espn_cfb_schedule(season=2024)
schedule.shape

In [ ]:
(schedule
 .select(['id', 'date', 'home_team_full_name', 'away_team_full_name', 'home_score', 'away_score'])
 .head())

## Play-by-play

`CFBPlayProcess(gameId=...)` drives the full ESPN college-football PBP pipeline: call `.espn_cfb_pbp()` to fetch the raw game summary, then `.run_processing_pipeline()` returns a dict whose `plays` key is the processed play list (EPA/WPA, down & distance, play types) alongside an `advBoxScore` and game/team metadata.

In [ ]:
from sportsdataverse.cfb import CFBPlayProcess

game = CFBPlayProcess(gameId=401628334)
game.espn_cfb_pbp()  # fetch the raw ESPN game summary
processed = game.run_processing_pipeline()  # full PBP feature pipeline (EPA/WPA + adv box score)
list(processed.keys())[:8]

In [ ]:
plays = pl.DataFrame(processed['plays'], infer_schema_length=None)
plays.select(['period', 'clock.displayValue', 'pos_team', 'down', 'distance', 'text', 'scoring_play', 'EPA']).head()

## Play participants (the `resolve_missing` flag)

`espn_cfb_play_participants` returns a per-play long frame of athletes who participated in each play. By default it falls back to the canonical ESPN `$ref` URL when the sidecar omits an athlete; set `resolve_missing=False` to skip that fan-out.

In [ ]:
participants = sdv.cfb.espn_cfb_play_participants(
    game_id=401628334,
    resolve_missing=True,
    resolve_missing_max=20,
)
participants.shape

In [ ]:
participants.head()

## Multi-season schedule via the loader

In [ ]:
schedule_2023 = sdv.cfb.load_cfb_schedule(seasons=[2023])
schedule_2023.shape

## Pipeline example: top 10 highest-scoring games of 2024

Combine schedule + simple polars expressions.

In [ ]:
(schedule
    .with_columns((pl.col('home_score') + pl.col('away_score')).alias('total_points'))
    .sort('total_points', descending=True)
    .select(['date', 'home_team_full_name', 'away_team_full_name', 'home_score', 'away_score', 'total_points'])
    .head(10))

## Cross-references

- R companion: [cfbfastR](https://cfbfastR.sportsdataverse.org)
- Data source: [ESPN CFB API](https://www.espn.com/college-football/)
- Plotting: [matplotlib](https://matplotlib.org), [plotnine](https://plotnine.org), or [polars-native plotting](https://docs.pola.rs)

## Where to go next

- API docs: `docs/docs/cfb/index.md`
- Next notebook: `03_nfl_intro.ipynb`